# Local & Open-Weight LLMs: Quantization, Tool Calling, and Hardware Benchmarks

**Objective:** Evaluate the feasibility of running open-weight Large Language Models (LLMs) on local consumer hardware. This analysis explores model distribution mechanisms (Hugging Face vs. Ollama), quantifies the RAM impact of precision compression (fp16 to int4), probes the reliability of tool calling in small parameter models, and benchmarks local throughput against hosted cloud APIs.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from local_models_utils import model_size_gb, fits_on, laptop_options, load_local_catalog, probe_tool_calling, speed_table
from providers_utils import groq_client, ollama_client, ask
from landscape_utils import time_stream, PROMPT

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")
try:
    ollama = ollama_client()
    local_models = [m.id for m in ollama.models.list().data]
    print("Ollama models available:", local_models)
except Exception as e:
    ollama, local_models = None, []
    print("Ollama not reachable:", e)

## 1. Open Weights Ecosystem & Hugging Face Distribution
Ollama packages and runs open-weight models originating from hubs like Hugging Face. Here we review the local catalog configuration.

In [ ]:
print(load_local_catalog())

**Key Finding — Gated Models:** A "gated" tag on a Hugging Face model refers to the distribution mechanism, not the legal license. It simply means the creator requires users to accept an Acceptable Use Policy or provide information to track download metrics before accessing the weights.

## 2. Quantization Mechanics & VRAM Budgeting
Model size is determined by parameters multiplied by bytes-per-parameter. To fit massive models on local hardware, we must compress the precision (quantization).

In [ ]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec in ("fp32", "fp16", "int8", "int4"):
    print(f"{prec:10} {'':>12} {model_size_gb(20, prec):>12.1f} GB")

In [ ]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.
MY_RAM_GB = 16
fits = laptop_options(MY_RAM_GB)
print(f"On a {MY_RAM_GB} GB laptop, these fit (with 2 GB headroom for everything else):")
for m in fits:
    print(f"  {m['name']:22} {model_size_gb(m['params_billion'], m['precision']):5.1f} GB   tool calling: {m['tool_calling']}")

**Key Finding — The Necessity of INT4:** We must use int4 quantization instead of fp16 because local consumer hardware (laptops) has severely limited memory. By compressing the model weights from 16 bits per parameter down to 4 bits, we drastically reduce the VRAM/RAM footprint, allowing complex models to actually fit and execute on local machines without out-of-memory errors.

## 3. Evaluating Function & Tool Calling Support
Not every small model handles deterministic function calling reliably. An agentic workflow requires a model that can emit strictly formatted JSON tool calls rather than conversational text.

In [ ]:
if ollama and local_models:
    for name in local_models[:3]:
        r = probe_tool_calling(ollama, name)
        print(f"{name:20} supports_tool_calling={r['supports_tool_calling']}")
        if not r["supports_tool_calling"]:
            print("   said instead:", (r.get('raw_text') or '')[:150])
else:
    print("skipped (Ollama not running)")

**Key Finding — Agentic Fallbacks:** If the current local model fails the tool-calling probe, we must alter the deployment architecture. We can either upgrade to a larger local model (provided the hardware has sufficient RAM), or we must offload the structured agentic workflow to an external cloud API model while reserving the cheaper local model strictly for standard conversational inference tasks.

## 4. Latency & Throughput: Local Compute vs. Hosted Cloud
We benchmark the exact same prompt against the Groq API and our local hardware to measure the true discrepancy in tokens per second.

In [ ]:
results = {}
try:
    results["groq"] = time_stream(groq, GROQ_MODEL)
except Exception as e:
    print("groq failed:", e)
if ollama and local_models:
    try:
        results["ollama"] = time_stream(ollama, local_models[0])
    except Exception as e:
        print("ollama failed:", e)
else:
    print("ollama skipped (not running)")

if results:
    print(speed_table(results))

**Key Finding — Workload Latency Sensitivity:** Local throughput is significantly slower than cloud APIs, but throughput speed matters least for the document reader workload because it operates as a background batch process. In contrast, the FAQ bot and agent-assist workflows require ultra-low latency because human users are actively waiting for real-time answers. Thus, local deployments are highly optimal for asynchronous data pipelines but risk causing friction in live conversational UI.